In [ ]:
import os
import time
import datetime

import xlwings as xw
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    ElementNotInteractableException,
    StaleElementReferenceException,
)
import datetime
import re
from pathlib import Path
import pandas as pd

start_range = [
    '01.07.2026', '01.01.2026', '01.07.2025', '01.01.2025', '01.07.2024', '01.01.2024', 
    '01.07.2023', '01.01.2023', '01.07.2022', '01.01.2022', '01.07.2021'
]

fin_range = [
    str((datetime.datetime.today().date() - datetime.timedelta(days=2)).strftime("%d.%m.%Y")), '30.06.2026', '31.12.2025', '30.06.2025', '31.12.2024', '30.06.2024', 
    '31.12.2023', '30.06.2023', '31.12.2022', '30.06.2022', '31.12.2021'
]
# определяем фильтры
ne_opredeleno = [['Не определено']]
akcii_invfnds = [
    ["Акции"],
    ["банки и финансовые институты",
    "высокая капитализация",
    "дивидендные акции",
    "не определено",
    "не распределено (архив)",
    "средняя капитализация",
    "широкий рынок",]
]
obligi = [ ['Облигации'], ['esg', 'все секторы', 'высокодоходные', 'государственные', 'корпоративные']]
# smeshanni = [['Смешанный'], ['не определено']]
# nedviga = [['Недвижимость'], ['не определено']]
# denejny = [['Денежный'], ['не определено']]
# dragmetalli = [['Драгметаллы'], ['золото', 'корзина драгметаллов', 'серебро']]
# ne_opredeleno = [['Не определено']] # мусорный фильтр, он пуст

all_spect_we_request = {}
all_spect_we_request['akcii_invfunds'] = akcii_invfnds
all_spect_we_request['obligi'] = obligi
# all_spect_we_request['smeshanni'] = smeshanni
# all_spect_we_request['nedviga'] = nedviga
# all_spect_we_request['denejny'] = denejny
# all_spect_we_request['dragmetalli'] = dragmetalli


import os
import time
import datetime

import xlwings as xw
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    ElementNotInteractableException,
    StaleElementReferenceException,
)

_TAIL = re.compile(r"\s*\[[^\]]*\]\s*$")   # хвост «[Акции]» в подписях spec_check_list

import datetime
import os
import re
from pathlib import Path

import pandas as pd

# функция очищает excel, в которой сшиты несколько других экселей от всякого ужаса
def strip_inner_headers(
    path,
    out_path=None,
    header_values=("Дата", "Фонд", "УК", "Пай", "СЧА"),
    drop_empty_rows=True,
    date_format="DD.MM.YYYY",
    verbose=True,
):
    """
    Чистит уже готовую книгу: на каждом листе оставляет шапку только в первой строке,
    все её повторы внутри данных (они приезжают из каждого скачанного файла) удаляет.

    path         : путь к финальной книге. Лучше абсолютный: относительный Python
                   считает от текущей папки ядра, а она не обязана совпадать с той,
                   куда Excel положил книгу.
    out_path     : куда писать; по умолчанию рядом файл с суффиксом '_clean'.
                   Передайте тот же path, чтобы перезаписать оригинал — пишем во
                   временный файл рядом и подменяем им исходный только в конце,
                   так что оригинал цел, даже если чистка упадёт на полпути.
    header_values: подписи шапки. None — взять шапкой первую непустую строку листа.
                   Сравнение без регистра, лишних пробелов и пустых ячеек.
    drop_empty_rows: заодно выкинуть полностью пустые строки (стыки блоков).
    date_format  : формат для ячеек-дат в готовой книге; None — не трогать.

    Возвращает {лист: {'было', 'шапок_удалено', 'пустых_удалено', 'стало'}}.
    Данные не трогает: удаляются только строки, целиком совпавшие с шапкой.
    """
    src = Path(path)
    dst = Path(out_path) if out_path else src.with_name(src.stem + "_clean" + src.suffix)

    def norm(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ""
        return re.sub(r"\s+", " ", str(v).replace("\xa0", " ")).strip().lower()

    def row_key(values):                      # компактный ключ строки без пустых ячеек
        return "|".join(x for x in (norm(v) for v in values) if x)

    def clean_sheet(df, sheet):
        """Отдаёт (очищенный кадр, отчёт по листу)."""
        was = len(df)
        if df.empty:                          # лист без данных переносим как есть
            return df, dict(было=0, шапок_удалено=0, пустых_удалено=0, стало=0)

        key = None                            # ключ строки одним проходом по столбцам
        for i in range(df.shape[1]):
            col = df.iloc[:, i].map(norm)
            key = col if key is None else key + "|" + col
        key = key.str.replace(r"\|{2,}", "|", regex=True).str.strip("|")

        hdr = want
        if hdr is None:                       # шапка не задана — первая непустая строка
            nonempty = key[key != ""]
            hdr = nonempty.iloc[0] if len(nonempty) else ""
        if re.search(r"\d", hdr):             # страховка: в шапке цифр не бывает
            raise ValueError(f"{src.name}[{sheet}]: подозрительная шапка {hdr!r}")

        # hdr == "" бывает только на листе без единой непустой строки — шапки там нет
        is_hdr = (key == hdr) if hdr else pd.Series(False, index=key.index)
        keep_first = is_hdr.idxmax() if is_hdr.any() else None   # верхнюю шапку сохраняем
        drop = is_hdr.copy()
        if keep_first is not None:
            drop.loc[keep_first] = False
        empty = (key == "") & ~drop
        if drop_empty_rows:
            drop |= empty

        out = df.loc[~drop]
        assert len(out) == was - int(drop.sum()), "потеряли строки"
        return out, dict(
            было=was,
            шапок_удалено=int(is_hdr.sum()) - (keep_first is not None),
            пустых_удалено=int(empty.sum()) if drop_empty_rows else 0,
            стало=len(out),
        )

# функция для работы с форматом даты 
    def apply_date_format(ws, out):
        """Иначе даты в готовой книге выглядят как '2025-07-01 00:00:00'."""
        probe = out.head(50)
        for j in range(out.shape[1]):
            is_dt = probe.iloc[:, j].map(lambda v: isinstance(v, datetime.datetime))
            if not is_dt.any():
                continue
            for (cell,) in ws.iter_rows(min_col=j + 1, max_col=j + 1):
                if isinstance(cell.value, datetime.datetime):
                    cell.number_format = date_format

    want = row_key(header_values) if header_values else None
    report = {}

    # Пишем в tmp рядом с dst и подменяем в конце: на Windows файл, который читает
    # openpyxl, нельзя одновременно открыть на запись (WinError 32) — а значит
    # out_path=path иначе не работал бы; заодно оригинал не станет обрубком, если
    # что-то упадёт по дороге.
    tmp = dst.with_name(dst.stem + ".__tmp__" + dst.suffix)
    try:
        with pd.ExcelFile(src, engine="openpyxl") as xls, pd.ExcelWriter(
            tmp, engine="openpyxl", datetime_format=date_format
        ) as writer:
            try:
                for sheet in xls.sheet_names:
                    out, rep = clean_sheet(
                        xls.parse(sheet, header=None, dtype=object), sheet
                    )
                    out.to_excel(writer, sheet_name=sheet, header=False, index=False)
                    if date_format and not out.empty:
                        apply_date_format(writer.sheets[sheet], out)

                    report[sheet] = rep
                    if verbose:
                        print(f"{src.name}[{sheet}]: {rep['было']} -> {rep['стало']} строк "
                              f"(шапок {rep['шапок_удалено']}, пустых {rep['пустых_удалено']})")
            except BaseException:
                # книгу без единого листа openpyxl сохранить не может и на выходе из
                # with подменил бы нашу ошибку своим IndexError
                if not writer.book.sheetnames:
                    writer.book.create_sheet("_")
                raise

        os.replace(tmp, dst)
    finally:
        if tmp.exists():
            tmp.unlink()

    if verbose:
        print(f"записано: {dst}")
    return report

# --- как пользоваться -------------------------------------------------------
# Путь только абсолютный и только после common_wb.save(...) + common_wb.close():
# до save книга живёт лишь в памяти Excel, файла на диске ещё нет, а голое имя
# pandas ищет от текущей папки ядра Jupyter — это и есть FileNotFoundError.
#
#     COMMON_WB_PATH = os.path.join(BASE_DIR, key_batch + os.path.basename(elem))
#     common_wb.save(COMMON_WB_PATH)
#     common_wb.close()
#     strip_inner_headers(COMMON_WB_PATH, out_path=COMMON_WB_PATH)
#
# Или отдельным проходом по тому, что реально лежит в папке:
#     import glob
#     for f in sorted(glob.glob(os.path.join(BASE_DIR, "*ПИФЫ.xlsx"))):
#         strip_inner_headers(f, out_path=f)


def _norm(text: str, keep_group: bool) -> str:
    t = (text or "").replace("\xa0", " ").strip().lower()
    if not keep_group:
        t = _TAIL.sub("", t)
    return re.sub(r"\s+", " ", t).strip()


def _force_click(driver, el, cb_id):
    """Нативный клик -> клик по label -> JS-клик. Возвращает способ, которым дошло."""
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
    except Exception:
        pass

    try:
        el.click()
        return "native"
    except (ElementClickInterceptedException, ElementNotInteractableException,
            StaleElementReferenceException):
        pass

    try:  # label[for] форвардит клик на input и триггерит его onclick
        driver.find_element(By.CSS_SELECTOR, f"label[for='{cb_id}']").click()
        return "label"
    except Exception:
        pass

    # HTMLElement.click() переключает checked И выполняет inline onclick=setspecs()/setuk(),
    # работает даже на display:none. В отличие от присваивания .checked, которое событий не шлёт.
    driver.execute_script("arguments[0].click();", driver.find_element(By.ID, cb_id))
    return "js"

# функция выбирает на сайте investfunds pro нужные нам фильтры
def click_filter(driver, wait, name, container_id="cat_check_list",
                 state=None, click_all=True, settle=0.3):
    """
    Кликает чекбокс фильтра по подписи label внутри нужного контейнера — независимо
    от того, виден ли он физически (скролл контейнера, схлопнутый блок, перехват клика).

    driver, wait : активный WebDriver и WebDriverWait
    name         : имя из списка-фильтра. Регистр не важен, хвост «[Категория]» можно
                   не писать («акции роста» == «Акции роста [Акции]»). Если скобки в
                   name есть — матч по полной подписи, что снимает неоднозначность
                   «Не определено [Смешанный]» vs «Не определено [Денежный]».
    container_id : "cat_check_list" (категории, c<N>) либо "spec_check_list" (специализации, s<N>)
    state        : None — просто переключить; True/False — довести до нужного состояния
                   (идемпотентно, не зависит от истории итераций; is_selected() читает
                   свойство и работает даже для невидимого элемента)
    click_all    : кликать все совпадения — в spec_check_list есть дубли подписей
                   с разными id (s34 и s67 «потребительский сектор [Акции]»)
    settle       : пауза после клика — onclick=setspecs()/setuk() перерисовывает списки

    Возвращает список (id, способ_клика). ValueError, если совпадений нет.
    """
    keep_group = "[" in name
    target = _norm(name, keep_group)

    ids = []
    for lbl in driver.find_elements(By.CSS_SELECTOR, f"#{container_id} label[for]"):
        try:
            if _norm(lbl.get_attribute("textContent"), keep_group) == target:
                cb_id = lbl.get_attribute("for")
                if cb_id and cb_id not in ids:
                    ids.append(cb_id)
        except StaleElementReferenceException:
            continue

    if not ids:
        raise ValueError(f"{name!r} не найден в #{container_id}")
    if not click_all:
        ids = ids[:1]

    clicked = []
    for cb_id in ids:
        # presence, а не element_to_be_clickable: элемент может быть в DOM, но скрыт
        box = wait.until(EC.presence_of_element_located((By.ID, cb_id)))
        was = box.is_selected()
        if state is not None and was == state:
            continue

        how = _force_click(driver, box, cb_id)
        if settle:
            time.sleep(settle)

        box = driver.find_element(By.ID, cb_id)   # мог быть перерисован обработчиком
        if box.is_selected() == was:              # клик не дошёл — добиваем через JS
            driver.execute_script("arguments[0].click();", box)
            if settle:
                time.sleep(settle)
            box = driver.find_element(By.ID, cb_id)
            how = "js-retry"
            if box.is_selected() == was:
                raise RuntimeError(f"состояние {cb_id} ({name!r}) не изменилось после клика")

        clicked.append((cb_id, how))
    return clicked

def _labels_map(driver, container_id):
    """{id: (полная_подпись, база_без_[группы])} по всем чекбоксам контейнера."""
    out = {}
    for lbl in driver.find_elements(By.CSS_SELECTOR, f"#{container_id} label[for]"):
        try:
            cb_id, txt = lbl.get_attribute("for"), lbl.get_attribute("textContent")
        except StaleElementReferenceException:
            continue
        if cb_id:
            out[cb_id] = (_norm(txt, True), _norm(txt, False))
    return out


def _checked_ids(driver, container_id):
    return {b.get_attribute("id")
            for b in driver.find_elements(
                By.CSS_SELECTOR, f"#{container_id} input[type=checkbox]")
            if b.is_selected()}

# снимает ВСЕ фильтры на сайте
def clear_filters(driver, wait, container_ids=("cat_check_list", "spec_check_list"),
                  settle=0.4, fast=True, attempts=2, strict=True):
    """
    Снимает все отмеченные чекбоксы в указанных контейнерах (на свежей странице
    отмечено всё). Один JS-проход на контейнер + добивание WebDriver-кликами.

    container_ids : строка или кортеж; порядок важен — см. замечание ниже
    fast          : снимать одним execute_script без round-trip'ов на каждый чекбокс
    attempts      : сколько раз повторить проход, если после перерисовки что-то вернулось
    strict        : RuntimeError, если снять всё так и не удалось

    Возвращает {container_id: сколько чекбоксов осталось отмеченными} (в норме нули).
    """
    if isinstance(container_ids, str):
        container_ids = (container_ids,)

    residual = {}
    for cid in container_ids:
        try:
            wait.until(lambda d, c=cid: d.find_elements(
                By.CSS_SELECTOR, f"#{c} input[type=checkbox]"))
        except TimeoutError:          # контейнера нет/пуст — например, spec после сброса категорий
            residual[cid] = 0
            continue

        for _ in range(attempts):
            if fast:
                driver.execute_script(
                    "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                    ".forEach(function (b) { if (b.checked) b.click(); });",
                    "#" + cid,
                )

def select_only(driver, wait, names, container_id="cat_check_list",
                settle=0.4, fast=True, strict=True):
    """
    Приводит контейнер к состоянию «отмечены ровно names»:
      1) снимает все отмеченные (на свежей странице отмечены все);
      2) ставит перечисленные в names;
      3) сверяет фактическое состояние с ожидаемым и падает при расхождении.

    names        : список имён из вашего list specializacii (регистр и хвост
                   «[Категория]» необязательны; с хвостом — точный матч)
    container_id : "cat_check_list" | "spec_check_list"
    fast         : снимать всё одним JS-проходом (18 кликов без round-trip'ов
                   WebDriver'а вместо 18 отдельных команд)
    strict       : ValueError, если какое-то имя из names не найдено в контейнере

    Возвращает отсортированный список id, оставшихся отмеченными.
    """
    wait.until(lambda d: d.find_elements(By.CSS_SELECTOR, f"#{container_id} label[for]"))

    # --- 1. снять всё
    clear_filters(driver, wait, container_id, settle=settle, fast=fast)

    # --- 2. поставить нужные
    labels = _labels_map(driver, container_id)   # список мог быть перерисован шагом 1
    want, missing = [], []
    for n in names:
        keep_group = "[" in n
        t = _norm(n, keep_group)
        hits = [i for i, (full, base) in labels.items()
                if (full if keep_group else base) == t]
        want.extend(hits) if hits else missing.append(n)
    if missing and strict:
        raise ValueError(f"не найдены в #{container_id}: {missing}")

    want = sorted(set(want))
    for cb_id in want:
        box = wait.until(EC.presence_of_element_located((By.ID, cb_id)))
        if not box.is_selected():
            _force_click(driver, box, cb_id)
            time.sleep(settle)

    # --- 3. верификация: молчаливо неверный фильтр хуже исключения
    got = _checked_ids(driver, container_id)
    if got != set(want):
        raise RuntimeError(
            f"#{container_id}: ожидали {want}, получили {sorted(got)}"
        )
    return want

"""
Выгрузка котировок с pro.investfunds.ru по полугодовым окнам
+ сборка всех скачанных файлов в одну «общую» книгу через xlwings.

Логика по каждому периоду:
  1. Selenium подставляет даты, жмёт кнопку XLS.
  2. Ждём, пока файл реально догрузится (нет .crdownload, размер стабилен).
  3. xlwings открывает файл, берёт блок = Ctrl+A из ANCHOR (C17),
     копирует его и вставляет в общую книгу под уже накопленные строки.
  4. Исходная книга закрывается без сохранения.
"""


# ---------------------------------------------------------------- конфигурация

BASE_DIR = r"C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0" # директория venv. Настрой под себя, с толчка она не работает.
DRIVER_PATH = os.path.join(BASE_DIR, "msedgedriver.exe") # сюда же нужно положить драйвер edge. Edge сам по себе нужно предварительно скачать.
DOWNLOAD_DIR = os.path.join(BASE_DIR, "downloads")   # куда Edge кладёт xls
COMMON_WB_PATHS = [os.path.join(BASE_DIR, "ОПИФЫ.xlsx"), os.path.join(BASE_DIR, "БПИФЫ.xlsx"), os.path.join(BASE_DIR, "ИПИФЫ.xlsx")]
READY_DIR = os.path.join(BASE_DIR, "ready_files")    # сюда кладём вычищенные книги

ANCHOR = "C4"          # точка, из которой «нажимаем Ctrl+A»
SKIP_HEADER_ROWS = 0    # сколько верхних строк блока пропускать со 2-го файла
PERIODS_PER_SHEET = 15  # сколько периодов ложится на один лист общей книги
DOWNLOAD_TIMEOUT = 120  # сек на одну выгрузку

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(READY_DIR, exist_ok=True)    # без неё сохранение упадёт на первой же книге

# я выше уже указал старт и финиш
# start_range = [
#     '01.07.2026', '01.01.2026'
# ]
# fin_range = [
#     str(datetime.datetime.today().date().strftime("%d.%m.%Y")), '30.06.2026'
# ]
N_PERIODS = len(start_range)
# ------------------------------------------------------------------- утилиты ФС

PARTIAL_SUFFIXES = (".crdownload", ".tmp", ".partial")


def dir_snapshot(folder: str) -> set:
    return set(os.listdir(folder))


def wait_for_new_download(folder: str, before: set, timeout: int = DOWNLOAD_TIMEOUT) -> str:
    """Ждёт появления нового полностью догруженного файла и возвращает его путь."""
    deadline = time.time() + timeout
    last_size = -1
    while time.time() < deadline:
        fresh = [
            f for f in os.listdir(folder)
            if f not in before and not f.endswith(PARTIAL_SUFFIXES)
        ]
        if fresh:
            path = max(
                (os.path.join(folder, f) for f in fresh),
                key=os.path.getmtime,
            )
            size = os.path.getsize(path)
            if size > 0 and size == last_size:   # два замера подряд совпали
                return path
            last_size = size
        time.sleep(0.7)
    raise TimeoutError(f"Новый файл не появился в {folder} за {timeout} с")


# --------------------------------------------------------------- утилиты Excel

def target_sheet(wb: xw.Book, period_idx: int) -> xw.Sheet:
    """
    Лист-приёмник для периода period_idx: первые PERIODS_PER_SHEET периодов —
    'общая', следующие — 'общая_2', и т.д. Лист создаётся при первом обращении.
    """
    sheet_idx = period_idx // PERIODS_PER_SHEET
    name = "общая" if sheet_idx == 0 else f"общая_{sheet_idx + 1}"
    if name in (s.name for s in wb.sheets):
        return wb.sheets[name]
    return wb.sheets.add(name, after=wb.sheets[-1])


def append_block(app: xw.App, src_path: str, dst_sheet: xw.Sheet, start_row: int) -> int:
    """
    Открывает src_path, берёт current_region от ANCHOR (= Ctrl+A из C17),
    копирует через буфер и вставляет в dst_sheet начиная со start_row.
    Возвращает число вставленных строк.
    """
    wb = app.books.open(src_path, update_links=False, read_only=True)
    try:
        sht = wb.sheets[0]
        block = sht.range(ANCHOR).current_region

        if start_row > 1 and SKIP_HEADER_ROWS:
            block = (
                block.offset(row_offset=SKIP_HEADER_ROWS)
                     .resize(block.rows.count - SKIP_HEADER_ROWS, block.columns.count)
            )

        n_rows = block.rows.count
        block.copy()                                  # Ctrl+C
        dst_sheet.range((start_row, 1)).paste()        # Ctrl+V
        app.api.CutCopyMode = False                    # снять «бегущий пунктир»
        return n_rows
    finally:
        wb.close()


# ------------------------------------------------------------------ Selenium

edge_options = webdriver.EdgeOptions()
# edge_options.add_argument("--headless=new")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--use-system-proxy-resolver")
user_data_dir = os.path.expanduser("~") + r"\AppData\Local\Microsoft\User Data"
edge_options.add_argument(f"--user-data-dir={user_data_dir}")
edge_options.add_argument("--profile-directory=Default")
edge_options.add_experimental_option(
    "prefs",
    {
        "download.default_directory": DOWNLOAD_DIR,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    },
)

driver = webdriver.Edge(
    options=edge_options,
    service=Service(executable_path=DRIVER_PATH),
)
wait = WebDriverWait(driver, 20)
try:
    driver.get("https://pro.investfunds.ru/")
    driver.get("https://pro.investfunds.ru/quotes/")
    ln = driver.find_element(By.ID, "loginName")
    ln.click()
    ln.send_keys("baxa")  # ввод логина
    pw = driver.find_element(By.ID, "loginPass")
    pw.click()
    pw.send_keys("mozart")  # ввод пароля
    driver.find_element(
        By.CSS_SELECTOR, "input.header-user-login-submit[type='submit']"
    ).click()
    app = xw.App(visible=False, add_book=False)
    app.display_alerts = False
    app.screen_updating = False
    for key_batch, value_batch in all_spect_we_request.items():
        print(f"Тянем {key_batch}")
        for elem in COMMON_WB_PATHS:
            next_rows = {}   # имя листа -> следующая свободная строка на нём

            common_wb = app.books.add()
            common_sht = common_wb.sheets[0]
            common_sht.name = "общая"
            # --- логин

            COMMON_WB_PATH = elem
            proxy_to_match = os.path.basename(COMMON_WB_PATH)

            clear_filters(driver, wait, container_ids=['type_check_list'])
            match proxy_to_match: #  Мне показалось это лучший вариант реализации парсера
                # опифы - триггер t1
                # бпифы - триггер t4
                # ипифы - триггер t2
                # зпифы - триггер t3
                case "ОПИФЫ.xlsx":
                    wait.until(EC.element_to_be_clickable((By.ID, "t1"))).click()
                case "БПИФЫ.xlsx":
                    wait.until(EC.element_to_be_clickable((By.ID, "t4"))).click()
                case "ИПИФЫ.xlsx":
                    wait.until(EC.element_to_be_clickable((By.ID, "t2"))).click()

            select_only(driver=driver, wait=wait, names=value_batch[0], container_id='cat_check_list')
            select_only(driver=driver, wait=wait, names=value_batch[1], container_id ="spec_check_list", strict=False)

            for i in range(N_PERIODS):
                st_dat = driver.find_element(By.ID, "calendar-1")
                fin_dat = driver.find_element(By.ID, "calendar-2")

                st_dat.send_keys(Keys.CONTROL + "a")
                st_dat.send_keys(Keys.BACKSPACE)
                st_dat.send_keys(start_range[i])

                fin_dat.send_keys(Keys.CONTROL + "a")
                fin_dat.send_keys(Keys.BACKSPACE)
                fin_dat.send_keys(fin_range[i])
                fin_dat.send_keys(Keys.ESCAPE)   # закрыть всплывающий календарь

                time.sleep(1)  # дать таблице пересобраться под новый диапазон

                before = dir_snapshot(DOWNLOAD_DIR)
                driver.find_element(By.ID, "xls").click()
                downloaded = wait_for_new_download(DOWNLOAD_DIR, before)

                dst = target_sheet(common_wb, i)
                start_row = next_rows.get(dst.name, 1)
                rows = append_block(app, downloaded, dst, start_row)
                next_rows[dst.name] = start_row + rows

                print(f"[{i + 1}/{N_PERIODS}] {start_range[i]}–{fin_range[i]}: "
                    f"{rows} строк -> {dst.name}!A{start_row} "
                    f"({os.path.basename(downloaded)})")

            total = sum(v - 1 for v in next_rows.values())
            name = key_batch + os.path.basename(COMMON_WB_PATH)   # 'obligiОПИФЫ.xlsx'
            # абсолютный путь: относительное имя Excel и pandas резолвят по-разному
            COMMON_WB_PATH = os.path.join(BASE_DIR, name)
            ready_path = os.path.join(READY_DIR, name)

            common_wb.save(COMMON_WB_PATH)
            common_wb.close()          # чистим только после close: до save файла нет,
            strip_inner_headers(COMMON_WB_PATH, out_path=ready_path)   # а Excel держит его
            print(f"Готово: {ready_path}, листов {len(next_rows)}, всего строк {total}")

    print()
finally:
    app.quit()
    driver.quit()